In [ ]:
import csv, os
from datetime import datetime

ARCHIVO = "finanzas.csv"
CATS = ["Alimentación", "Transporte", "Arriendo", "Salud", "Educación", "Entretenimiento", "Otros"]

def cargar():
    if not os.path.exists(ARCHIVO):
        return []
    with open(ARCHIVO, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def guardar(tipo, cat, desc, monto):
    nuevo = not os.path.exists(ARCHIVO)
    with open(ARCHIVO, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["fecha","tipo","categoria","descripcion","monto"])
        if nuevo: w.writeheader()
        w.writerow({"fecha": datetime.now().strftime("%Y-%m-%d"), "tipo": tipo,
                    "categoria": cat, "descripcion": desc, "monto": monto})
    print(f"✓ {tipo.capitalize()} de ${float(monto):,.0f} guardado.\n")

def elegir_cat():
    for i, c in enumerate(CATS, 1): print(f"  {i}. {c}")
    while True:
        try:
            op = int(input("Categoría: "))
            if 1 <= op <= len(CATS): return CATS[op - 1]
        except ValueError: pass

def pedir_monto():
    while True:
        try:
            m = float(input("Monto: $").replace(",","."))
            if m > 0: return m
        except ValueError: pass

def resumen():
    datos = cargar()
    if not datos: return print("Sin datos aún.\n")
    ing = sum(float(t["monto"]) for t in datos if t["tipo"] == "ingreso")
    gas = sum(float(t["monto"]) for t in datos if t["tipo"] == "gasto")
    print(f"\n  Ingresos : ${ing:>10,.0f}")
    print(f"  Gastos   : ${gas:>10,.0f}")
    print(f"  Balance  : ${ing-gas:>10,.0f}  {'✓' if ing>=gas else '✗'}\n")

def por_categoria():
    datos = [t for t in cargar() if t["tipo"] == "gasto"]
    if not datos: return print("Sin gastos.\n")
    totales = {}
    for t in datos: totales[t["categoria"]] = totales.get(t["categoria"], 0) + float(t["monto"])
    total = sum(totales.values())
    print()
    for cat, m in sorted(totales.items(), key=lambda x: -x[1]):
        print(f"  {cat:<16} ${m:>9,.0f}  {m/total*100:5.1f}%")
    print(f"\n  {'TOTAL':<16} ${total:>9,.0f}\n")

def historial():
    datos = cargar()[-10:]
    if not datos: return print("Sin datos aún.\n")
    print(f"\n{'Fecha':<12}{'Tipo':<9}{'Categoría':<16}{'Monto':>10}  Descripción")
    print("-" * 60)
    for t in reversed(datos):
        s = "+" if t["tipo"] == "ingreso" else "-"
        print(f"{t['fecha']:<12}{t['tipo']:<9}{t['categoria']:<16}{s}${float(t['monto']):>8,.0f}  {t['descripcion']}")
    print()

def menu():
    opciones = {"1":"Registrar ingreso","2":"Registrar gasto","3":"Resumen",
                "4":"Por categoría","5":"Historial","6":"Salir"}
    while True:
        print("── Finanzas Personales ──")
        for k, v in opciones.items(): print(f"  {k}. {v}")
        op = input("\nOpción: ").strip()
        if op in ("1","2"):
            tipo = "ingreso" if op == "1" else "gasto"
            cat  = elegir_cat()
            desc = input("Descripción: ").strip() or "Sin descripción"
            guardar(tipo, cat, desc, pedir_monto())
        elif op == "3": resumen()
        elif op == "4": por_categoria()
        elif op == "5": historial()
        elif op == "6": print("¡Hasta luego!"); break
        else: print("Opción inválida.\n")

menu()